# fesom-jax tutorial — config, run, output

A tour of the **release run path**: a single `RunConfig` YAML drives a run, the driver chunks it in
time and honours a dt-ramp, and the streaming accumulator gives the time mean / variance (and the EKE
map) without storing every step. The cells here are **data-free** (no mesh needed) so they run
anywhere; the final section shows the one command that launches an actual model run on Levante.

See `docs/USER_GUIDE.md` for the full YAML schema.

## 1. Load and inspect a config

The shipped `configs/ng5.yaml` pins the NG5 physics (GM off, `visc_gamma1=0.2`, a dt 180→240 ramp,
the implemented MFCT/QR4C/FCT tracer scheme). Unknown keys raise; absent keys take the bit-identical
default.

In [ ]:
from pathlib import Path
from fesom_jax.run_config import load_yaml

root = Path.cwd().parents[0] if Path.cwd().name == 'examples' else Path.cwd()
cfg = load_yaml(root / 'configs' / 'ng5.yaml')
print('GM:', cfg.gm, '| visc.gamma1:', cfg.visc.gamma1, '| dt:', cfg.dt)
print('dt_ramp:', cfg.dt_ramp, '| tracer (implemented):', cfg.tracer)

## 2. Run lengths and time-chunking

`parse_duration` turns `"2yr"`/`"3mo"`/`"5d"`/`"10step"` into a step count; `plan_chunks` splits a run
into fine time-chunks (so a multi-year forcing is never pre-stacked) AND at the dt-ramp boundary,
tagging each chunk with its dt and whether it bootstraps AB2 (cold / post-ramp) or continues it.

In [ ]:
from fesom_jax.run import parse_duration, plan_chunks

n = parse_duration('1d', cfg.dt)          # one day at dt=180 s
print('1 day =', n, 'steps')
for c in plan_chunks(n, chunk_steps=240, start_step=0, dt_ramp=cfg.dt_ramp, dt=cfg.dt):
    print(c)

## 3. Streaming mean / variance / EKE

`OnlineStats` (Welford) accumulates the per-grid-point time mean **and** variance over a run without
storing every step — so the mean state and the eddy-kinetic-energy map come for free.

In [ ]:
import numpy as np
from fesom_jax.zarr_output import OnlineStats, eke_from_stats

rng = np.random.default_rng(0)
stats = None
for _ in range(50):                         # pretend these are per-step surface velocity fields
    uv = {'uv': rng.standard_normal((100, 1, 2))}
    stats = OnlineStats.init(uv) if stats is None else stats.update(uv)
print('mean shape:', stats.mean_dict()['uv'].shape, '| EKE mean:', float(eke_from_stats(stats).mean()))

## 4. Launch an actual run (on Levante, with the data)

The data-free cells above exercise the *machinery*. To run the model, point the driver at a staged mesh
+ forcing and let it load → run → write a portable restart:

```bash
# one segment; resume by adding --restart-in DIR (any device count):
python scripts/run_from_config.py configs/core2_full.yaml --steps 480 --restart-out runs/core2/seg0

# a chained multi-segment campaign on SLURM (de-risk on farc/dars before NG5):
scripts/chain_submit.sh configs/dars.yaml 6 4800 /work/.../runs/dars
```

Read a field back from the gather-free Zarr output to a dense global array with
`fesom_jax.zarr_output.reconstruct_global(out_dir, 'T')` and plot it. See `docs/USER_GUIDE.md`.